# Experiment 4: Latency Tradeoff

Inspect the benchmark of eight questions: two per populated retrieval subgroup. Document embedding and index preparation are outside the timed paths.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.


**Shared question scope:** all four experiments select from the frozen **15,585 fully covered question IDs**. Targeted reranking uses **500 opportunities + 200 controls**; answer quality samples **up to 25 per resulting subgroup**, and latency samples from those answers. Passage embeddings are reused; live question embedding, reranking, generation and judging calls produce outputs in this notebook.

**Workflow:** Experiment 1 (Baseline) --> Diagnosis --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> **this Experiment 4 (Latency Tradeoff)**


## 1. Set up the latency study

**What:** prepare the end-to-end benchmark and standard scoring.

**Why:** extend the v3 answer-quality story with the operational cost of a complete reranked path.

Use the root pinned `requirements.txt`. Live execution requires `OPENROUTER_API_KEY`, prepared document vectors and the FAISS index. Model settings and portable artifact paths live in `config.json`. Optional new benchmarks and plot exports use `requirements-latency.txt`.

In [1]:
STUDY = 'latency'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


Setting,Value
Study,Experiment 4: Latency Tradeoff
Setting,Value
Benchmark run ID,latency-tradeoff-fresh-a8ff6898f0612c5123ef
Corpus passages,152576
Query embedding,qwen/qwen3-embedding-4b · 2560 dimensions
Reranker,qwen/qwen3-reranker-8b
Generator,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Generation limit,400 tokens
Workers,1


## 2. Keep the frozen sample and run order

The frozen benchmark sample has **eight questions**, sampled with seed 43: two each from opportunity gain, unchanged, loss, and all-support controls. The fifth planned subgroup has no eligible cases; it is not pending work.

The source answer-quality sample has 79 questions. Sampled IDs precede benchmark generation and scoring. Arms run sequentially with one worker, alternating which arm runs first by sample position.

These deliberately stratified eight pairs are a descriptive operational check, not a population estimate. Two cases per populated group cannot support stable group-level quality conclusions.

The benchmark contains eight question pairs. The empty fifth subgroup has no eligible cases and is not a pending stage.

In [2]:
view.show('questions')
if MODE != "plan":
    assignments = pd.read_parquet(tutorial.upstream("latency") / "input/questions.parquet")
    display_table(assignments[["query_id", "latency_sample_order", "arm_order"]])


Stratum,Question pairs
Opportunity gain,2.00
Opportunity unchanged,2.00
Opportunity loss,2.00
Control: all support already present,2.00
Control: some support absent from top 30,Not applicable (0 eligible questions)


latency_sample_order,query_id,generation_stratum,question,First path
0,1046747,control_all_support_already_present,what is rx provigil u,Embedding RAG
1,106620,control_all_support_already_present,cost of repainting cabinets,Embedding + reranking RAG
2,1062919,opportunity_gain,what is an exeval,Embedding RAG
3,1084140,opportunity_gain,what does book mean for utilities,Embedding + reranking RAG
4,1079518,opportunity_loss,what groups benefit from social programs,Embedding RAG
5,1100471,opportunity_loss,foods that unsaturated,Embedding + reranking RAG
6,1083354,opportunity_unchanged,what does nodc stand for in state gov,Embedding RAG
7,1084423,opportunity_unchanged,what does a person on probation have to do,Embedding + reranking RAG


## 3. Define the complete timed paths

**What:** each path made its own query-embedding request, retrieved 30 passages, selected five and generated an answer. The reranked path added a request to rerank its own exact 30 candidates.

**Why:** measure the operational cost of adding reranking to a full RAG path.

Corpus, precomputed document embeddings, FAISS settings, generator settings and passage budgets were fixed. Document embedding and one-time index/corpus loading were excluded from path timing. A persistent HTTP session served the sequential run.

These are the benchmark’s own answers and contexts. Their earlier v3 counterparts are separate outputs. Each path’s exact request bodies, passages, responses and stage durations remain inspectable.

**What changed from Experiment 3?** Experiment 3 uses frozen top-five contexts. This benchmark made fresh query-embedding and generation requests for each arm, so it measured the complete online path. The eight question IDs are frozen; each path retrieves its own candidates. The reranker sees exactly its path’s retrieved 30 passages. Do not assume both paths or the earlier study have identical candidate sets without comparing their passage IDs.

| Work | Timed? | Why? |
|---|---|---|
| Document embeddings and index/corpus loading | No | Precomputed or loaded before the sequential benchmark. |
| Fresh query embedding and local retrieval | Yes, in both arms | Required to answer each incoming question. |
| Reranking the retrieved 30 passages | Yes, reranked arm only | The extra evidence-selection stage. |
| Select five passages, build prompt and generate | Yes, in both arms | Complete the response. |
| Offline answer scoring | No | Evaluation happens after the response. |


In [3]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))

# Serial online paths, counterbalanced arm order. New HTTP calls and retries are timed.
# Corpus/index loading is outside measured path time. Cached HTTP responses cannot
# be substituted into an incomplete benchmark's timings.
benchmark_run = tutorial.benchmark(embed_api, rerank_api, generate_api)
if benchmark_run is not None:
    display(benchmark_run)
view.show('completion')


Path,Work in each path
Embedding RAG,"[query_embedding, generation]"
Embedding + reranking RAG,"[query_embedding, rerank_30, generation]"


## 4. Separate timing scopes and check coverage

**What:** distinguish successful-request time, successful-path time and total wall time.

**Why:** a retry can dominate the user’s wait even when the eventual successful request is fast.

**Successful path** adds the successful HTTP attempts to local retrieval and prompt selection. **Total wall time** includes all attempts, actual backoff, local work and client overhead. All durations use seconds unless a table explicitly labels milliseconds.

The Qwen benchmark has **16 successful complete paths (eight pairs)**, **40 hosted stages**, and **41 timed attempts**. One timed failure and its retry are retained. There are **zero excluded setup records** and **zero truncated answers**.

| Timing scope | Plain description | What it helps explain |
|---|---|---|
| Successful-request time | Add up only the successful hosted attempts across the path. | Hosted work after excluding failed attempts. |
| Successful-path time | Successful-request time plus local compute. | The path’s measured work excluding failures and backoff. |
| Total end-to-end wall time | All attempts, backoff, local work and client overhead. | The full measured wait for the completed answer. |

This benchmark used non-streaming generation: it measures completion time, not time to first token. A successful-request total is a sum across several stages, not the duration of one generation call.


In [4]:
view.show('completion')
view.show('attempts')


Timing field,Definition
successful_path_seconds,Successful hosted requests plus local work
total_end_to_end_seconds,"All attempts, actual backoff and local work"


Coverage,Count
Complete path pairs,8
Saved answers,16
Timed hosted stages,40
Timed hosted attempts,41
Timed failed attempts (retained),1
Recorded retries,1
Excluded setup records (retained separately),0
Truncated answers (retained),0


## 5. Compare complete-path latency

These are completed Qwen measurements for eight paired questions. Compare within-question deltas, retaining failed attempts and actual backoff in total wall time. Network and serving-backend variation are part of these observations; they are not a production latency guarantee. No timing finding is imported from the earlier model setup.


In [5]:
view.show('latency')


Path,Complete paths,Retries,Median successful path (s),Median total wall time (s),p95 total wall time (s)
Embedding RAG,8,0,3.504,3.525,8.004
Embedding + reranking RAG,8,1,3.956,4.441,6.714


Timing scope,Matching pairs,Median paired added time (s)
Successful path,8,0.31
Total wall time,8,0.32


arm,questions,retries,median_successful_request_seconds,p95_successful_request_seconds,median_successful_path_seconds,p95_successful_path_seconds,median_total_end_to_end_seconds,p95_total_end_to_end_seconds,median_retry_overhead_seconds
baseline,8,0,3.458831,7.932908,3.504003,7.980152,3.524906,8.004349,0.020768
reranked,8,1,3.913788,6.358523,3.955978,6.401000,4.440543,6.714420,0.032114


query_id,successful_request_seconds_baseline,successful_request_seconds_reranked,successful_path_seconds_baseline,successful_path_seconds_reranked,total_end_to_end_seconds_baseline,total_end_to_end_seconds_reranked,successful_request_seconds_added_by_reranking,successful_path_seconds_added_by_reranking,total_end_to_end_seconds_added_by_reranking,question,generation_stratum,latency_sample_order,arm_order
1046747,8.743530,6.644348,8.792233,6.689740,8.819237,6.721583,-2.099182,-2.102493,-2.097655,what is rx provigil u,control_all_support_already_present,0,"[baseline, reranked]"
106620,2.077505,1.990976,2.119491,2.035614,2.138906,2.066165,-0.086529,-0.083877,-0.072741,cost of repainting cabinets,control_all_support_already_present,1,"[reranked, baseline]"
1062919,1.766098,3.567448,1.811863,3.610543,1.832876,3.637847,1.801350,1.798681,1.804971,what is an exeval,opportunity_gain,2,"[baseline, reranked]"
1084140,6.427467,3.389319,6.472002,3.435031,6.490984,3.466296,-3.038148,-3.036971,-3.024688,what does book mean for utilities,opportunity_gain,3,"[reranked, baseline]"
1079518,3.310963,3.588384,3.356174,3.630245,3.376697,3.665344,0.277422,0.274070,0.288648,what groups benefit from social programs,opportunity_loss,4,"[baseline, reranked]"
1100471,3.606700,5.137966,3.651831,5.182967,3.673115,5.215741,1.531266,1.531136,1.542626,foods that unsaturated,opportunity_loss,5,"[reranked, baseline]"
1083354,1.914161,4.239192,1.956522,4.281711,1.977919,6.701119,2.325032,2.325189,4.723200,what does nodc stand for in state gov,opportunity_unchanged,6,"[baseline, reranked]"
1084423,5.485675,5.827705,5.526762,5.864767,5.546255,5.897152,0.342030,0.338005,0.350898,what does a person on probation have to do,opportunity_unchanged,7,"[reranked, baseline]"


query_id,question,generation_stratum,latency_sample_order,arm,arm_position,status,started_utc,completed_utc,successful_request_seconds,hosted_all_attempt_seconds,backoff_seconds,local_compute_seconds,successful_path_seconds,total_end_to_end_seconds,retry_count,answer,finish_reason,generation_usage,served_model,system_fingerprint,retrieved_top30_passage_ids,selected_top5_passage_ids
1046747,what is rx provigil u,control_all_support_already_present,0,baseline,1,success,2026-10-15T22:58:02.336745+00:00,2026-10-15T22:58:11.155987+00:00,8.743530,8.743530,0.000000,0.048703,8.792233,8.819237,0,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,"{'prompt_tokens': 557, 'completion_tokens': 221, 'total_tokens': 778, 'cost': 0.00011643, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00011643, 'upstream_inference_prompt_cost': 5.013e-05, 'upstream_inference_completions_cost': 6.63e-05}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}",qwen/qwen3-30b-a3b-instruct-2507,None,"[p_24a53cfbfc767599e0c0, p_fda5caf24b89ebe2509a, p_9af703a77c5788aa82b2, p_4db7d78a64f92cba650c, p_7110f1d3ec0ebf07c6b8, p_76e35b87c3ed87314368, p_86a92124e6f610137602, p_d791e60f70b2d386d93e, p_29faa5b5abcfc7df1720, p_3a175c1e172821beba5c, p_3fd6b5e0758b4c17d23f, p_95b9ab497a9cf13a8963, p_e4df629b5e90c9b81a4d, p_2c813bd17d8f21944622, p_6f8545faa4b1a3c589cf, p_e517c869567b8dbef182, p_6c4f9bab490440839803, p_3f9276bbf1778d29433e, p_87a262fafacf6f614496, p_bf5d8ff085a812c02f11, p_ae7b8b6b6505b79cb086, p_c2d9546602c0f095dbad, p_52787022b8837bcdedd3, p_3aa5d6659a4fee929b36, p_8efa5a0cb6de1660b9b1, p_f1ed0a8f6c1ba4ef21b7, p_a223fd7eab0670cc692a, p_6160d1aea7b099de5b9b, p_31e572d906a01336a758, p_849771bfda3ca5db0f8a]","[p_24a53cfbfc767599e0c0, p_fda5caf24b89ebe2509a, p_9af703a77c5788aa82b2, p_4db7d78a64f92cba650c, p_7110f1d3ec0ebf07c6b8]"
query_id,question,generation_stratum,latency_sample_order,arm,arm_position,status,started_utc,completed_utc,successful_request_seconds,hosted_all_attempt_seconds,backoff_seconds,local_compute_seconds,successful_path_seconds,total_end_to_end_seconds,retry_count,answer,finish_reason,generation_usage,served_model,system_fingerprint,retrieved_top30_passage_ids,selected_top5_passage_ids
1046747,what is rx provigil u,control_all_support_already_present,0,reranked,2,success,2026-10-15T22:58:11.165851+00:00,2026-10-15T22:58:17.887434+00:00,6.644348,6.644348,0.000000,0.045393,6.689740,6.721583,0,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are excessively sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness and is classified as a nonstimulant or stimulant medication, though its exact mechanism is not fully understood. It is not approved for use in children under 17 years of age, and it may interact with certain medications, including hormonal contraceptives, and is not recommended for long-term use or in individuals with liver problems [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_7110f1d3ec0ebf07c6b8][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c].",st

In [6]:
if MODE != "plan":
    times = pd.read_csv(tutorial.folder / "latency.csv", dtype={"query_id": str})
    paired_wall_time = times.pivot(index="query_id", columns="arm", values="total_end_to_end_seconds").dropna()
    paired_wall_time["reranked_minus_baseline_seconds"] = paired_wall_time.reranked - paired_wall_time.baseline
    display_table(paired_wall_time)


arm,query_id,baseline,reranked,reranked_minus_baseline_seconds
,1046747,8.8192,6.7216,-2.0977
,1062919,1.8329,3.6378,1.8050
,106620,2.1389,2.0662,-0.0727
,1079518,3.3767,3.6653,0.2886
,1083354,1.9779,6.7011,4.7232
,1084140,6.4910,3.4663,-3.0247
,1084423,5.5463,5.8972,0.3509
,1100471,3.6731,5.2157,1.5426


## 6. Identify the stage costs

**What:** separate query embedding, local retrieval, reranking, selection and generation.

**Why:** the reranker request’s own cost is only one part of the complete path difference.

The hosted reranker stage has a median of **0.487 seconds**. The full paired difference also includes each path’s embedding, generation and local-work variation. Generation dominates the observed variability.

This table labels stage durations in **milliseconds**. Expand the **40 hosted stage measurements** for exact **seconds**, per-path attempt counts and backoff.

**Component time is not added path time.** Use the rebuilt run’s query embedding, reranking, generation and retry timings. Stage medians should not be added to reconstruct a path median.


In [7]:
view.show('latency_stages')


Path,Stage,calls,Median (ms),p95 (ms),retries
Embedding RAG,Generate answer,8,3098.24,6375.54,0
Path,Stage,calls,Median (ms),p95 (ms),retries
Embedding RAG,Query embedding,8,513.42,2017.65,0
Embedding RAG,Retrieve 30 candidates,8,44.82,47.67,0
Embedding RAG,Select five and build prompt,8,0.01,0.01,0
Embedding + reranking RAG,Generate answer,8,2936.89,5116.46,0
Embedding + reranking RAG,Query embedding,8,476.13,1228.65,0
Embedding + reranking RAG,Rerank 30 candidates,8,486.58,2641.03,1
Embedding + reranking RAG,Retrieve 30 candidates,8,43.84,45.58,0
Embedding + reranking RAG,Select five and build prompt,8,0.02,0.04,0


query_id,arm,arm_position,stage,stage_type,status,attempt_count,successful_request_seconds,all_attempt_seconds,backoff_seconds,stage_total_seconds
1046747,baseline,1,generation,hosted_request,success,1,7.451167,7.451167,0.000000,7.459775
query_id,arm,arm_position,stage,stage_type,status,attempt_count,successful_request_seconds,all_attempt_seconds,backoff_seconds,stage_total_seconds
1046747,baseline,1,query_embedding,hosted_request,success,1,1.292363,1.292363,0.000000,1.310487
1046747,baseline,1,retrieve_30_seconds,local,success,0,0.000000,0.000000,0.000000,0.048695
1046747,baseline,1,select_and_build_prompt_seconds,local,success,0,0.000000,0.000000,0.000000,0.000009
1046747,reranked,2,generation,hosted_request,success,1,5.463511,5.463511,0.000000,5.474494
1046747,reranked,2,query_embedding,hosted_request,success,1,0.598889,0.598889,0.000000,0.610045
1046747,reranked,2,rerank_30,hosted_request,success,1,0.581949,0.581949,0.000000,0.591132
1046747,reranked,2,retrieve_30_seconds,local,success,0,0.000000,0.000000,0.000000,0.045367
1046747,reranked,2,select_and_build_prompt_seconds,local,success,0,0.000000,0.000000,0.000000,0.000025


## 7. Inspect the retained failure and retry

The failure is **query 1083354, reranked arm, rerank_30**. Its first attempt lasted about **0.35 seconds**, followed by about **2.01 seconds** of backoff and a successful retry. Failed-attempt time and backoff remain in total wall time; successful-path time excludes them.

There are **zero excluded setup records**. The empty display below retains the old caption “The two excluded setup/implementation failure records”; it does not establish two such failures in this Qwen run.


In [8]:
view.show('failures')
view.show('attempts')


query_id,arm,stage,attempt,elapsed_seconds,backoff_after_seconds
1083354,reranked,rerank_30,1,0.35,2.01


query_id,arm,arm_position,stage,attempt,started_utc,status,status_code,elapsed_seconds,error,backoff_after_seconds
1046747,baseline,1,generation,1,2026-10-15T22:58:03.700391+00:00,success,200,7.451167,Pending,0.000000
query_id,arm,arm_position,stage,attempt,started_utc,status,status_code,elapsed_seconds,error,backoff_after_seconds
1046747,baseline,1,query_embedding,1,2026-10-15T22:58:02.349302+00:00,success,200,1.292363,Pending,0.000000
1046747,reranked,2,generation,1,2026-10-15T22:58:12.419096+00:00,success,200,5.463511,Pending,0.000000
1046747,reranked,2,query_embedding,1,2026-10-15T22:58:11.171276+00:00,success,200,0.598889,Pending,0.000000
1046747,reranked,2,rerank_30,1,2026-10-15T22:58:11.826086+00:00,success,200,0.581949,Pending,0.000000
1062919,baseline,1,generation,1,2026-10-15T22:58:22.681632+00:00,success,200,1.270775,Pending,0.000000
1062919,baseline,1,query_embedding,1,2026-10-15T22:58:22.128693+00:00,success,200,0.495323,Pending,0.000000
1062919,reranked,2,generation,1,2026-10-15T22:58:24.786805+00:00,success,200,2.812932,Pending,0.000000
1062919,reranked,2,query_embedding,1,2026-10-15T22:58:23.970678+00:00,success,200,0.234094,Pending,0.000000


## 8. The answer-scoring

**What:** score the 16 answers produced by the timed benchmark with the same libraries used in v3.

**Why:** compare latency with quality on the same question, answer and arm—not with an answer from another run.

Our preprocessing removes passage-ID citations while preserving answer text. Hugging Face SQuAD then applies its normalization and best-reference scoring across every retained reference. **Token F1 and normalized exact match use 0–100 percentages**. The custom-scorer comparison uses **0–1**.

RAGAS **Faithfulness** uses only that arm’s exact five benchmark passages. **Answer relevancy** uses the generated answer and question, with 2,560-dimensional Qwen evaluation embeddings. Their query instruction matches retrieval; evaluation results and caches are rebuilt separately.

Both RAGAS metrics are complete for all eight pairs, with independently reported pair counts and failures. Judge estimates are distinct from human correctness. Evaluation latency is outside the RAG path timings.


In [9]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))


Metric,Scale,What it measures
SQuAD normalized exact match,0–100 percent,Does the normalized answer equal any reference answer?
Token F1,0–100 percent,Best normalized token overlap across all reference answers.
Faithfulness,0–1,Are answer claims supported by this arm’s exact five saved passages?
Answer relevancy,usually 0–1; cosine can be −1–1,Does the saved answer address the original question?


## 9. Run answer scoring

**What:** compute SQuAD scores and RAGAS judgments for the benchmark’s 16 answers.

**Why:** compare latency and quality for the exact same timed answers.

Hugging Face `evaluate.load` and multiple-reference `metric.compute` provide local SQuAD scoring. The official SQuAD source is available locally.

RAGAS uses the real `llm_factory`, `Faithfulness` and `AnswerRelevancy` entry points. It requires `OPENROUTER_API_KEY` and logs prompts, settings, responses, failures, capped retries and timing. Evaluation happens outside the timed online paths. Inspect the implementations in the appendix.

In [ ]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output("Show remaining latency scoring output — expand/collapse", preview_rows=1, preview_logs=True):
    score_run = await tutorial.score()
    if score_run is not None:
        display(score_run)
    view.show('judge_outcomes')


First scoring entry visible; expand for all remaining latency answer-scoring output.

## 10. Inspect the latency–quality tradeoff

The numeric tables join each of eight within-question latency deltas with Token F1 for the exact timed answers. Positive latency delta means the reranked path took longer; positive F1 delta means more reference-word overlap.

If no plot is available, the numeric comparisons remain inspectable. If a plot is exported later, label it as eight pairs. No model calls are needed to render measured results.

Each populated subgroup has only two pairs. Inspect individual cases rather than treating subgroup bootstrap intervals as stable population evidence. Annotated-support Recall@5 stays separate on its 0–1 scale. Use the larger Experiment 3 for answer-quality analysis.


In [11]:
view.show('latency_quality')
view.show('scores')


Saved plot unavailable. The numeric tables remain available; export a plot explicitly if needed.

query_id,Stratum,Successful path delta (s),Total wall-time delta (s),Token F1 delta (percentage points)
1046747,Control: all support already present,-2.10,-2.10,1.84
106620,Control: all support already present,-0.08,-0.07,8.33
1062919,Opportunity gain,1.80,1.80,-2.83
1084140,Opportunity gain,-3.04,-3.02,4.51
1079518,Opportunity loss,0.27,0.29,4.15
1100471,Opportunity loss,1.53,1.54,-7.80
1083354,Opportunity unchanged,2.33,4.72,0.00
1084423,Opportunity unchanged,0.34,0.35,10.35


query_id,question,generation_stratum,Successful path delta (s),Total wall-time delta (s),Token F1 delta (percentage points),Stratum
1046747,what is rx provigil u,control_all_support_already_present,-2.102493,-2.097655,1.841949,Control: all support already present
106620,cost of repainting cabinets,control_all_support_already_present,-0.083877,-0.072741,8.333333,Control: all support already present
1062919,what is an exeval,opportunity_gain,1.798681,1.804971,-2.831403,Opportunity gain
1084140,what does book mean for utilities,opportunity_gain,-3.036971,-3.024688,4.506803,Opportunity gain
1079518,what groups benefit from social programs,opportunity_loss,0.274070,0.288648,4.146730,Opportunity loss
1100471,foods that unsaturated,opportunity_loss,1.531136,1.542626,-7.801418,Opportunity loss
1083354,what does nodc stand for in state gov,opportunity_unchanged,2.325189,4.723200,0.000000,Opportunity unchanged
1084423,what does a person on probation have to do,opportunity_unchanged,0.338005,0.350898,10.354374,Opportunity unchanged


Stratum,Successful matching pairs,Embedding RAG (%),Embedding + reranking RAG (%),Paired delta (percentage points)
Opportunity gain,2,17.69,18.52,0.84
Opportunity unchanged,2,58.62,63.80,5.18
Opportunity loss,2,23.68,21.86,-1.83
Control: all support already present,2,14.52,19.61,5.09
Control: some support absent from top 30,0,Not applicable (0 eligible questions),Not applicable (0 eligible questions),Not applicable (0 eligible questions)


Stratum,Metric,Successful answers,Failed answers,Pending answers,Successful matching pairs
Opportunity gain,Faithfulness,4,0,0,2
Stratum,Metric,Successful answers,Failed answers,Pending answers,Successful matching pairs
Opportunity unchanged,Faithfulness,4,0,0,2
Opportunity loss,Faithfulness,4,0,0,2
Control: all support already present,Faithfulness,4,0,0,2
Control: some support absent from top 30,Faithfulness,0,0,0,0
Opportunity gain,Answer relevancy,4,0,0,2
Opportunity unchanged,Answer relevancy,4,0,0,2
Opportunity loss,Answer relevancy,4,0,0,2
Control: all support already present,Answer relevancy,4,0,0,2


generation_stratum,Metric,Scale,Successful matching pairs,Embedding RAG,Embedding + reranking RAG,Paired delta (reranking − embedding),Improved,Tied,Lower,Delta bootstrap 95% low,Delta bootstrap 95% high
opportunity_gain,SQuAD normalized exact match,0–100 percent,2,0.0,0.0,0.0,0,2,0,0.000000,0.000000
generation_stratum,Metric,Scale,Successful matching pairs,Embedding RAG,Embedding + reranking RAG,Paired delta (reranking − embedding),Improved,Tied,Lower,Delta bootstrap 95% low,Delta bootstrap 95% high
opportunity_gain,Token F1,0–100 percent,2,17.687075,18.524775,0.8377,1,0,1,-2.831403,4.506803
opportunity_gain,Faithfulness,0–1,2,0.9375,1.0,0.0625,1,1,0,0.000000,0.125000
opportunity_gain,Answer relevancy,usually 0–1; cosine can be −1–1,2,0.332809,0.6912,0.35839,2,0,0,0.117915,0.598866
opportunity_unchanged,SQuAD normalized exact match,0–100 percent,2,0.0,0.0,0.0,0,2,0,0.000000,0.000000
opportunity_unchanged,Token F1,0–100 percent,2,58.618331,63.795518,5.177187,1,1,0,0.000000,10.354374
opportunity_unchanged,Faithfulness,0–1,2,1.0,1.0,0.0,0,2,0,0.000000,0.000000
opportunity_unchanged,Answer relevancy,usually 0–1; cosine can be −1–1,2,0.849261,0.849261,0.0,0,2,0,0.000000,0.000000
opportunity_loss,SQuAD normalized exact match,0–100 percent,2,0.0,0.0,0.0,0,2,0,0.000000,0.000000


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness_0_1,faithfulness_status,faithfulness_seconds,answer_relevancy_0_1,answer_relevancy_status,answer_relevancy_seconds,cohort,batch_id,generation_stratum,is_baseline_diagnostic
1046747,what is rx provigil u,baseline,Embedding RAG,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c][p_7110f1d3ec0ebf07c6b8].",stop,False,1.000000,0.040404,0.0,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are very sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness. It is available in a generic form and is not approved for use in children under 17 years of age. It may interact with certain medications, should not be used long-term due to potential habit-forming effects, and is not recommended for individuals with liver problems or those using hormonal contraception .","[""Provigil is a stimulant that helps you feel awake.""]",0.0,4.040404,0.000000e+00,0.0,1.000000,ok,22.550091,0.600257,ok,12.508429,control,1,control_all_support_already_present,False
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness_0_1,faithfulness_status,faithfulness_seconds,answer_relevancy_0_1,answer_relevancy_status,answer_relevancy_seconds,cohort,batch_id,generation_stratum,is_baseline_diagnostic
1046747,what is rx provigil u,reranked,Embedding + reranking RAG,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are excessively sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness and is classified as a nonstimulant or stimulant medication, though its exact mechanism is not fully understood. It is not approved for use in children under 17 years of age, and it may interact with certain medications, including hormonal contraceptives, and is not recommended for long-term use or in individuals with liver problems [p_24a53cfbfc767599e0c0][p_fda5caf24b89ebe2509a][p_7110f1d3ec0ebf07c6b8][p_9af703a77c5788aa82b2][p_4db7d78a64f92cba650c].",stop,False,1.000000,0.058824,0.0,"Provigil (modafinil) is a prescription medication used to improve wakefulness in adults who are excessively sleepy due to sleep disorders such as narcolepsy, sleep apnea, or shift work sleep disorder. It is thought to work by altering neurotransmitters in the brain to increase alertness and is classified as a nonstimulant or stimulant medication, though its exact mechanism is not fully understood. It is not approved for use in children under 17 years of age, and it may interact with certain medications, including hormonal contraceptives, and is not recommended for long-term use or in individuals with liver problems .","[""Provigil is a stimulant that helps you feel awake.""]",0.0,5.882353,-6.9388

Stratum,Metric,Scale,Successful matching pairs,Embedding RAG,Embedding + reranking RAG,Paired delta (reranking − embedding)
Opportunity gain,Annotated-support Recall@5,0–1; existing retrieval metric,2,0.00,1.00,1.00
Opportunity unchanged,Annotated-support Recall@5,0–1; existing retrieval metric,2,0.33,0.33,0.00
Opportunity loss,Annotated-support Recall@5,0–1; existing retrieval metric,2,0.50,0.00,-0.50
Control: all support already present,Annotated-support Recall@5,0–1; existing retrieval metric,2,1.00,1.00,0.00
Control: some support absent from top 30,Annotated-support Recall@5,0–1; existing retrieval metric,0,Not applicable (0 eligible questions),Not applicable (0 eligible questions),Not applicable (0 eligible questions)


## 11. Inspect one question with both paths

The example is **1046747, “what is rx provigil u”**. Both arms retain their own exact contexts, candidates, prompts, response records, and timing. In this pair, the same five passage IDs appear in different orders.

Total wall time is approximately **8.82 s → 6.72 s** and Token F1 is **4.04% → 5.88%**. The short reference and much longer answers illustrate reference-overlap sensitivity to wording and verbosity. A single faster reranked response is not evidence that adding a reranker reliably reduces latency.

Dataset passages contain potentially conflicting medical claims. Treat these as experimental outputs, not medical guidance; human correctness review remains pending. Exact request bodies were recorded; response displays are rendered from JSON, not necessarily original HTTP wire bytes.


In [12]:
view.show('walkthrough', query_id='1046747')


arm,arm_position,successful_path_seconds,total_end_to_end_seconds,retry_count,backoff_seconds
baseline,1,8.79,8.82,0,0.00
reranked,2,6.69,6.72,0,0.00


**Question 1046747:** what is rx provigil u

Path,Token F1 (%),Normalized exact match (%),finish_reason,truncated
Embedding RAG,4.04,0.00,stop,False
Embedding + reranking RAG,5.88,0.00,stop,False


**Embedding RAG**

**Embedding + reranking RAG**

id,text,source_url,truncated,embedding_rank,similarity_score
p_24a53cfbfc767599e0c0,"Provigil (modafinil) is a medication that promotes wakefulness. It is thought to work by altering the natural chemicals (neurotransmitters) in the brain. Provigil is used to treat excessive sleepiness caused by sleep apnea, narcolepsy, or shift work sleep disorder. Provigil may also be used for purposes not listed in this medication guide.",https://www.drugs.com/provigil.html,False,1,0.637174
id,text,source_url,truncated,embedding_rank,similarity_score
p_fda5caf24b89ebe2509a,"Provigil (modafinil) is a nonstimulant medication that improves alertness. Provigil (modafinil) is available in a cheaper, generic version. Provigil (modafinil) is good at decreasing excessive sleepiness in shift work sleep disorder. If you need to sleep in the daytime, this medication won't effect you.",https://www.iodine.com/drug/provigil,False,2,0.619194
p_9af703a77c5788aa82b2,"Provigil (modafinil) is a stimulant that helps you feel awake. Researchers don't know exactly how Provigil (modafinil) works, but it likely effects chemicals (neurotransmitters) in the brain to increase alertness.",https://www.iodine.com/drug/provigil,False,3,0.611749
p_4db7d78a64f92cba650c,"Provigil (modafinil) is effective in improving alertness, but it interacts with some medicines and shouldn't be used long-term. Provigil (modafinil) can be habit forming, so it shouldn't be used for long-term treatment. Provigil (modafinil) is not a good choice if you have liver problems. May not be a good option if you are taking hormonal contraception like birth control pills.",https://www.iodine.com/drug/provigil,False,4,0.604979
p_7110f1d3ec0ebf07c6b8,PROVIGIL is not approved for use in children for any medical condition. It is not known if PROVIGIL is safe or effective in children under 17 years of age. What is PROVIGIL? PROVIGIL is a prescription medicine used to improve wakefulness in adults who are very sleepy due to one of the following diagnosed sleep disorders: narcolepsy,https://www.rxlist.com/provigil-drug.htm,False,5,0.596441
p_76e35b87c3ed87314368,What is the most important information I should know about PROVIGIL? PROVIGIL may cause serious side effects including a serious rash or a serious allergic reaction that may affect parts of your body such as your liver or blood cells. Any of these may need to be treated in a hospital and may be life-threatening.,https://www.rxlist.com/provigil-drug.htm,False,6,0.589071
p_86a92124e6f610137602,"Learn about Provigil (Modafinil) may treat, uses, dosage, side effects, drug interactions, warnings, patient labeling, reviews, and related medications.",https://www.rxlist.com/provigil-drug.htm,False,7,0.568206
p_d791e60f70b2d386d93e,"1-888-571-3100. MedPro Rx is a leading provider of intravenous immune globulin (IVIg) and subcutaneous immune globulin (SCIg) therapy to patients with a range of medical conditions. what is immune globulin? Immune globulin (IgG or IVIg) is a solution of highly purified immune globulin G, derived from large pools of human plasma that contain antibodies against a broad spectrum of bacterial and viral agents.",http://www.medprorx.com/drug-therapy/immune-globulin-therapy/,False,8,0.520564
p_29faa5b5abcfc7df1720,"Common Provigil side effects may include: 1 headache, dizziness; 2 feeling nervous or anxious; 3 back pain; 4 nausea, diarrhea, upset stomach; 5 sleep problems (insomnia); or. 6 stuffy nose.",https://www.drugs.com/provigil.html,False,9,0.515186


id,text,source_url,truncated,embedding_rank,similarity_score
p_24a53cfbfc767599e0c0,"Provigil (modafinil) is a medication that promotes wakefulness. It is thought to work by altering the natural chemicals (neurotransmitters) in the brain. Provigil is used to treat excessive sleepiness caused by sleep apnea, narcolepsy, or shift work sleep disorder. Provigil may also be used for purposes not listed in this medication guide.",https://www.drugs.com/provigil.html,False,1,0.637174
id,text,source_url,truncated,embedding_rank,similarity_score
p_fda5caf24b89ebe2509a,"Provigil (modafinil) is a nonstimulant medication that improves alertness. Provigil (modafinil) is available in a cheaper, generic version. Provigil (modafinil) is good at decreasing excessive sleepiness in shift work sleep disorder. If you need to sleep in the daytime, this medication won't effect you.",https://www.iodine.com/drug/provigil,False,2,0.619194
p_9af703a77c5788aa82b2,"Provigil (modafinil) is a stimulant that helps you feel awake. Researchers don't know exactly how Provigil (modafinil) works, but it likely effects chemicals (neurotransmitters) in the brain to increase alertness.",https://www.iodine.com/drug/provigil,False,3,0.611749
p_4db7d78a64f92cba650c,"Provigil (modafinil) is effective in improving alertness, but it interacts with some medicines and shouldn't be used long-term. Provigil (modafinil) can be habit forming, so it shouldn't be used for long-term treatment. Provigil (modafinil) is not a good choice if you have liver problems. May not be a good option if you are taking hormonal contraception like birth control pills.",https://www.iodine.com/drug/provigil,False,4,0.604979
p_7110f1d3ec0ebf07c6b8,PROVIGIL is not approved for use in children for any medical condition. It is not known if PROVIGIL is safe or effective in children under 17 years of age. What is PROVIGIL? PROVIGIL is a prescription medicine used to improve wakefulness in adults who are very sleepy due to one of the following diagnosed sleep disorders: narcolepsy,https://www.rxlist.com/provigil-drug.htm,False,5,0.596441
p_76e35b87c3ed87314368,What is the most important information I should know about PROVIGIL? PROVIGIL may cause serious side effects including a serious rash or a serious allergic reaction that may affect parts of your body such as your liver or blood cells. Any of these may need to be treated in a hospital and may be life-threatening.,https://www.rxlist.com/provigil-drug.htm,False,6,0.589071
p_86a92124e6f610137602,"Learn about Provigil (Modafinil) may treat, uses, dosage, side effects, drug interactions, warnings, patient labeling, reviews, and related medications.",https://www.rxlist.com/provigil-drug.htm,False,7,0.568206
p_d791e60f70b2d386d93e,"1-888-571-3100. MedPro Rx is a leading provider of intravenous immune globulin (IVIg) and subcutaneous immune globulin (SCIg) therapy to patients with a range of medical conditions. what is immune globulin? Immune globulin (IgG or IVIg) is a solution of highly purified immune globulin G, derived from large pools of human plasma that contain antibodies against a broad spectrum of bacterial and viral agents.",http://www.medprorx.com/drug-therapy/immune-globulin-therapy/,False,8,0.520564
p_29faa5b5abcfc7df1720,"Common Provigil side effects may include: 1 headache, dizziness; 2 feeling nervous or anxious; 3 back pain; 4 nausea, diarrhea, upset stomach; 5 sleep problems (insomnia); or. 6 stuffy nose.",https://www.drugs.com/provigil.html,False,9,0.515186


## 12. Diagnostic membership

Neither the IMEI walkthrough **1053665** nor the original effervescent diagnostic **1100870** appears among the eight latency questions. No answers, scores or timings from those diagnostic displays are substituted into this benchmark.

The generic output below uses an empty configured `diagnostic_ids` list. Its message is not an independent audit of either ID; the eight-question sample table establishes membership.


In [13]:
view.show('diagnostic_membership')


**Baseline diagnostic:** none of the configured diagnostic IDs is in this saved latency sample.

## 13. Interpret the rebuilt findings

Compare paired latency changes with the scores of those exact timed answers. Report failures and retries from this run; old provider timing conclusions do not apply.


In [14]:
view.show('latency')
view.show('scores')
tutorial.show()


Metric,Compared answers,Differences above tolerance,Maximum absolute difference (0–1)
Token F1,16,0,0.00
Normalized exact match,16,0,0.00


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness_0_1,faithfulness_status,faithfulness_seconds,answer_relevancy_0_1,answer_relevancy_status,answer_relevancy_seconds,cohort,batch_id,generation_stratum,is_baseline_diagnostic


## Live benchmark execution

**What:** run fresh query embedding, retrieval, reranking and generation on the fixed latency sample.

**Why:** obtain measurements under your machine, network and endpoint conditions.

The live benchmark runs in section 3 by default. It requires the pinned dependencies, `OPENROUTER_API_KEY`, and the prepared corpus/index declared in `configs/tutorial_inputs.json`. Document embedding and index loading remain outside each timed path.

Measurements and per-attempt traces belong to the current run. Request caps and bounded retries apply. Scoring evaluates the benchmark’s own answers after the timed paths complete.

In [15]:
# The benchmark ran once in section 3. Inspect that run's measurements here.
# Start a new Live RUN_ID at setup to measure another run.
view.show('latency')


## Appendix: inspect the real pipeline and framework calls

**What:** read the actual benchmark path, hosted-request tracing and framework entry points.

**Why:** keep the tutorial’s API calls verifiable while setup and bulk execution stay in helpers. Displaying source performs no inference, retrieval, scoring or index loading.

In [16]:
display(Markdown("```python\n" + inspect.getsource(tutorial.benchmark) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(run_ragas_evaluation) + "\n```"))


## Appendix: verify the frozen measurement and scoring inputs

**What:** validate the benchmark and scoring inputs with a local integrity check.

**Why:** verify the sample, prompts, answers, separate contexts, request/response traces, timing sums, retries and paired deltas.

Machine-specific asset paths are supplied through environment configuration. Notebook presentation hashes are informational because ordinary edits can change notebook bytes. Frozen parent data and configuration remain required dependencies.

See the adjacent README for setup, the file map and limitations.

The local integrity check displays its return value without requesting hosted model calls.

In [17]:
view.show('audit')


Saved provenance and integrity reports (not rerun):


Stratum,Metric,Successful answers,Failed answers,Pending answers,Successful matching pairs
Opportunity gain,Faithfulness,4,0,0,2
Stratum,Metric,Successful answers,Failed answers,Pending answers,Successful matching pairs
Opportunity unchanged,Faithfulness,4,0,0,2
Opportunity loss,Faithfulness,4,0,0,2
Control: all support already present,Faithfulness,4,0,0,2
Control: some support absent from top 30,Faithfulness,0,0,0,0
Opportunity gain,Answer relevancy,4,0,0,2
Opportunity unchanged,Answer relevancy,4,0,0,2
Opportunity loss,Answer relevancy,4,0,0,2
Control: all support already present,Answer relevancy,4,0,0,2
